In [1]:
import deepchem as dc
import numpy as np
import pandas as pd
import os
import tempfile
import random
import warnings
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import glob
warnings.filterwarnings("ignore", category=UserWarning)

param_grid = {
    'n_estimators': [200, 300],
    'min_child_weight': [8, 16]
}

#param_grid = {                     # for bigger qm8
#    'n_estimators': [400,600],      
#    'min_child_weight': [8, 16]
#}

def df_to_ecfp_dataset(df_input, tasks):
    featurizer = dc.feat.CircularFingerprint(size=2048)
    loader = dc.data.CSVLoader(tasks=tasks, smiles_field='SMILES', featurizer=featurizer)
    temp_path = tempfile.mktemp(suffix=".csv")
    df_input.to_csv(temp_path, index=False)
    dataset = loader.create_dataset(temp_path)
    os.remove(temp_path)
    return dataset

def evaluate_xgb(model, dataset):
    y_true = dataset.y.flatten()
    y_pred = model.predict(dataset.X)
    return {
        'r2_score': r2_score(y_true, y_pred),
        'mae_score': mean_absolute_error(y_true, y_pred),
        'rms_score': mean_squared_error(y_true, y_pred, squared=False)
    }

all_results_sole = []
path = os.path.dirname(os.getcwd())
train_df = pd.read_csv(f'{path}/datasets/example_train.csv')
train_df_filtered = pd.read_csv(f'{path}/datasets/results/clean_3_3.csv').iloc[:, :2]
valid_df = pd.read_csv(f'{path}/datasets/example_valid.csv')
test_df = pd.read_csv(f'{path}/datasets/example_test.csv')
tasks = ['Labels']

train_dataset = df_to_ecfp_dataset(train_df, tasks)
train_filtered_dataset = df_to_ecfp_dataset(train_df_filtered, tasks)
valid_dataset = df_to_ecfp_dataset(valid_df, tasks)
test_dataset = df_to_ecfp_dataset(test_df, tasks)

train_dataset_dict = {'original': train_dataset, 'filtered': train_filtered_dataset}

all_results_sole = []

for dataset_type, current_train_dataset in train_dataset_dict.items():
    print(f"\n===== Training with {dataset_type} dataset =====")
    best_val_r2 = -np.inf
    best_model = None
    best_params = None

    for n_estimators in param_grid['n_estimators']:
        for min_child_weight in param_grid['min_child_weight']:
            model = XGBRegressor(
                n_estimators=n_estimators,
                min_child_weight=min_child_weight,
                random_state=42,
                tree_method='gpu_hist',
                verbosity=0,
                n_jobs=-1
            )
            model.fit(current_train_dataset.X, current_train_dataset.y.flatten())
            val_pred = model.predict(valid_dataset.X)
            val_r2 = r2_score(valid_dataset.y.flatten(), val_pred)

            print(f"→ XGB(n_estimators={n_estimators}, min_child_weight={min_child_weight}) → Val R² = {val_r2:.4f}")

            if val_r2 > best_val_r2:
                best_val_r2 = val_r2
                best_model = model
                best_params = {
                    'n_estimators': n_estimators,
                    'min_child_weight': min_child_weight
                }

    train_scores = evaluate_xgb(best_model, current_train_dataset)
    valid_scores = evaluate_xgb(best_model, valid_dataset)
    test_scores = evaluate_xgb(best_model, test_dataset)

    print(f"Dataset: {dataset_type}")
    print(f"Train: {train_scores}")
    print(f"Valid: {valid_scores}")
    print(f"Test : {test_scores}")

    all_results_sole.append({
        'dataset_type': dataset_type,
        'used_params': best_params,
        'train_scores': train_scores,
        'valid_scores': valid_scores,
        'test_scores': test_scores
    })

results_df = pd.DataFrame([{
    'dataset_type': res['dataset_type'],
    **res['used_params'],
    'train_r2': res['train_scores']['r2_score'],
    'train_mae': res['train_scores']['mae_score'],
    'train_rmse': res['train_scores']['rms_score'],
    'valid_r2': res['valid_scores']['r2_score'],
    'valid_mae': res['valid_scores']['mae_score'],
    'valid_rmse': res['valid_scores']['rms_score'],
    'test_r2': res['test_scores']['r2_score'],
    'test_mae': res['test_scores']['mae_score'],
    'test_rmse': res['test_scores']['rms_score']
} for res in all_results_sole])

results_df.to_csv(f"{path}/datasets/results/xgb_results.csv", index=False)

smiles_field is deprecated and will be removed in a future version of DeepChem.Use feature_field instead.
smiles_field is deprecated and will be removed in a future version of DeepChem.Use feature_field instead.
smiles_field is deprecated and will be removed in a future version of DeepChem.Use feature_field instead.
smiles_field is deprecated and will be removed in a future version of DeepChem.Use feature_field instead.



===== Training with original dataset =====
→ XGB(n_estimators=200, min_child_weight=8) → Val R² = 0.4454
→ XGB(n_estimators=200, min_child_weight=16) → Val R² = 0.4440
→ XGB(n_estimators=300, min_child_weight=8) → Val R² = 0.4376
→ XGB(n_estimators=300, min_child_weight=16) → Val R² = 0.4277
Dataset: original
Train: {'r2_score': 0.8671264010269493, 'mae_score': 0.9991121422982927, 'rms_score': 1.702373385344529}
Valid: {'r2_score': 0.4454410789858695, 'mae_score': 2.090814057600607, 'rms_score': 3.2839076921536963}
Test : {'r2_score': 0.6535350087544496, 'mae_score': 1.6444363627800576, 'rms_score': 2.4086978260552927}

===== Training with filtered dataset =====
→ XGB(n_estimators=200, min_child_weight=8) → Val R² = 0.4384
→ XGB(n_estimators=200, min_child_weight=16) → Val R² = 0.3811
→ XGB(n_estimators=300, min_child_weight=8) → Val R² = 0.4104
→ XGB(n_estimators=300, min_child_weight=16) → Val R² = 0.3573
Dataset: filtered
Train: {'r2_score': 0.867633231747301, 'mae_score': 0.952277